In [5]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel
from statsmodels.stats.power import TTestPower

questions = ["q1", "q2", "q3", "q4", "q5", "q6"]

eval1_files = [
    r"data/evaluation1_llama4scout.csv",
    r"data/evaluation1_OSS20REAL.csv",
    r"data/evaluation1_OSS120.csv"
]

eval2_files = [
    r"data/evaluation2_llama4scout.csv",
    r"data/evaluation2_OSS20.csv",
    r"data/evaluation2_OSS120.csv"
]

In [6]:
def average_csv_files(files):
    dfs = []

    for file in files:
        df = pd.read_csv(file)

        df = df[questions].apply(pd.to_numeric, errors="coerce")

        dfs.append(df)

    lengths = [len(df) for df in dfs]

    if len(set(lengths)) != 1:
        raise ValueError(f"Files do not have the same number of rows: {lengths}")

    avg_df = sum(dfs) / len(dfs)

    return avg_df

eval1_avg = average_csv_files(eval1_files)
eval2_avg = average_csv_files(eval2_files)

In [7]:
overall_eval1_avg = np.mean(eval1_avg, axis=1)
overall_eval2_avg = np.mean(eval2_avg, axis=1)
eval1_avg["overall"] = overall_eval1_avg
eval2_avg["overall"] = overall_eval2_avg

In [8]:
alpha = 0.01
target_power = 0.80
alternative = "two-sided"

power_analysis = TTestPower()

results = []

questions.append("overall")

for q in questions:
    paired = pd.DataFrame({
        "eval1": eval1_avg[q],
        "eval2": eval2_avg[q]
    }).dropna()

    differences = paired["eval1"] - paired["eval2"]

    mean_diff = differences.mean()
    sd_diff = differences.std(ddof=1)

    if sd_diff == 0 or mean_diff == 0:
        effect_size_dz = np.nan
        required_n = np.inf
    else:
        # Cohen's dz for Wilcoxon test (dz because it's paired, d would be for independant)
        effect_size_dz = mean_diff / sd_diff

        required_n = power_analysis.solve_power(
            effect_size=abs(effect_size_dz),
            alpha=alpha,
            power=target_power,
            alternative=alternative
        )

        required_n = np.ceil(required_n)

    t_stat, p_value = ttest_rel(
        paired["eval1"],
        paired["eval2"]
    )

    results.append({
        "question": q,
        "pilot_n": len(differences),
        "eval1_mean": paired["eval1"].mean(),
        "eval2_mean": paired["eval2"].mean(),
        "mean_difference_eval1_minus_eval2": mean_diff,
        "sd_difference": sd_diff,
        "effect_size_dz": effect_size_dz,
        "pilot_p_value": p_value,
        "estimated_required_n": required_n
    })

sample_size_results = pd.DataFrame(results)
sample_size_results = sample_size_results.round(decimals=5)
sample_size_results.to_csv(r"pilot_run.csv", index=False)

sample_size_results

,question,pilot_n,eval1_mean,eval2_mean,mean_difference_eval1_minus_eval2,sd_difference,effect_size_dz,pilot_p_value,estimated_required_n
0,q1,100,4.77667,4.55333,0.22333,0.33183,0.67303,0.0000,30.0
1,q2,100,4.91333,4.61333,0.30000,0.38925,0.77071,0.0000,24.0
2,q3,100,4.79333,4.23000,0.56333,0.48245,1.16764,0.0000,12.0
3,q4,100,4.91667,4.88000,0.03667,0.24570,0.14923,0.1388,528.0
4,q5,100,4.25000,4.04000,0.21000,0.40936,0.51300,0.0000,48.0
5,q6,100,4.95333,4.73000,0.22333,0.33520,0.66628,0.0000,30.0
6,overall,100,4.76722,4.50778,0.25944,0.24262,1.06935,0.0000,14.0
